# 🧠 Deep Learning Modeling (Analisis Sentimen & Emosi)

Notebook ini berfungsi untuk **melatih model Deep Learning** (misal LSTM atau IndoBERT) dengan memisahkan arsitektur dan loop pelatihannya ke dalam modul terpisah di `src/dl_model.py`.

Konsep pemisahan (Separation of Concerns) ini **merupakan struktur yang sama** dengan fase preprocessing sebelumnya, di mana kode logika tersimpan di file `.py`, sementara `notebook` hanya bertugas melakukan eksekusi (training, visualisasi, dan evaluasi hasil).

In [ ]:
# 1. Import library utama
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import ast # opsional, kalau diperlukan untuk parsing dari list literal

# 2. Import Modul dari "src/"
import sys
import pathlib
# Pastikan Python bisa mendeteksi root project supaya direktori 'src' terbaca
ROOT_DIR = pathlib.Path().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.dl_model import LSTMSentimentModel, TextClassificationDataset, train_model

print("Modul PyTorch dan Deep Learning dari 'src/' berhasil diimpor!")

## 🔄 1. Load Data (Dataset Bersih)
Data yang kita load berasal dari `cleaned_dataset.csv` hasil notebook 01.

In [ ]:
# Load dataset bersih
df = pd.read_csv("../data/clean/cleaned_dataset.csv")
df.head()

## ⚙️ 2. Setup Tokenisasi / Vektorisasi
Di sini kalian bisa membangun kamus (Vocabulary) integer berdasarkan dataset, atau menggunakan pre-trained embedding.

In [ ]:
# Contoh inisialisasi placeholder tokenisasi & DataLoader Pytorch
# (Di tahap ini silakan tulis logic mapping word -> index)
X_dummy_sequences = [[1, 2, 3], [4, 5, 0], [2, 1, 6]]
y_dummy_labels = [0, 1, 0]

# Membuat PyTorch Dataset menggunakan class buatan kita dari "src/"
train_dataset = TextClassificationDataset(X_dummy_sequences, y_dummy_labels)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

## 🏗️ 3. Inisiasi Model dari `src/dl_model.py`
Model yang besar dan kompleks kelasnya **TIDAK** ditulis di notebook ini, melainkan diimpor langsung.

In [ ]:
INPUT_DIM = 1000 # Contoh ukuran Kosakata (Vocabulary Size)
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 2   # Misalnya 2 untuk klasifikasi Sentimen (Positif/Negatif)
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5

# Panggil dari src
model = LSTMSentimentModel(
    vocab_size=INPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    n_layers=N_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout=DROPOUT
)

print("Arsitektur Model yang digunakan:")
print(model)

## 🏋️‍♂️ 4. Siklus Pelatihan (Training Loop)
Menggunakan *logic* pelatihan per-epoch yang juga sudah membungkus helper-nya dari file Python.

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = criterion.to(device)

N_EPOCHS = 5
for epoch in range(N_EPOCHS):
    # Kita memanggil _train_model_ wrapper function dari `src/dl_model.py`
    train_loss, train_acc = train_model(model, train_loader, optimizer, criterion, device)
    print(f"Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%")

## 💾 5. Save Model untuk Keperluan Deploy (Hugging Face)
Simpan statenya ke direktori `models/` agar bisa dikonsumsi oleh endpoint Gradio Deep Learning kita di apps.

In [ ]:
# torch.save(model.state_dict(), '../models/dl_sentiment_lstm.pt')
# print("Model berhasil disave ke folder /models!")